<h1>Atividade 4 -- Tópicos para Computação 1 -- 2026.1</h1>

- Escola Superior de Tecnologia
- Profa. Dra. Elloá B. Guedes (ebgcosta@uea.edu.br)
- www.elloaguedes.com
- github.com/elloa
- Data: 19 de março de 2026

<h3>Descrição</h3>
A atividade consiste em construir uma rede neural convolucional usando transfer learning da rede Inception V3 para classificar o Stanford Dogs Dataset, um dataset com imagens de 120 raças de cachorro de todo o mundo e mais de 20 mil imagens para treino e teste

<h3>Requisitos:</h3>

- Uso de Transfer Learning
- Uso de Taxa de Aprendizado Adaptativa
- Data Augmentation
- L2 regularizatiom
- Early Stopping
- Gráficos de loss/acurácia no treino e validação
- Métricas no conjunto de teste
- Elaboração de um conjunto de slides com os resultados dos experimentos e apresentação da arquitetura
- Monitoramento do tempo de treinamento
- Quantidade de parâmetros totais e ajustáveis

<h3>Equipe:</h3>

- Ana Flavia de Castro Segadillha da Silva
- Davi Aguiar Moreira
- Guilherme Goncalves Moraes
- Luiz Fernando Borges Brito
- Pedro Vitor Barros Maranhao

<h2>Bibliotecas</h2>

In [1]:
import os
import kagglehub
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torchvision.models import inception_v3, Inception_V3_Weights
import random
import shutil

In [2]:
print(torch.cuda.is_available())

False


<h2>Extração e filtragem do dataset</h2>

In [3]:
#Opção 1 importar do kagglehub
path = kagglehub.dataset_download("miljan/stanford-dogs-dataset-traintest")

print("Path to dataset files:", path)

#Criação de novos paths para acessar os dados de treino e teste

novo_path = os.path.join(path, "cropped")

test_path = os.path.join(novo_path, "test")

train_path = os.path.join(novo_path, "train")

valid_path = os.path.join(novo_path, "test")

Path to dataset files: C:\Users\pedro\.cache\kagglehub\datasets\miljan\stanford-dogs-dataset-traintest\versions\1


In [20]:
#Opção 2 pegar do servidor
base_origem = '/opt/topicos/data'
base_destino = os.path.expanduser('~/data')  #voltar para home do usuário

for pasta in ['train', 'valid', 'test']:
    origem = os.path.join(base_origem, pasta)
    destino = os.path.join(base_destino, pasta)

    if not os.path.exists(destino):
        shutil.copytree(origem, destino)

train_path = os.path.join(base_destino, 'train')
test_path  = os.path.join(base_destino, 'test')
valid_path = os.path.join(base_destino, 'valid')


FileNotFoundError: [WinError 3] O sistema não pode encontrar o caminho especificado: '/opt/topicos/data\\train'

In [17]:
#Função para retirar "n[id]" do início das pastas buscando por "-"
def tirar_id(path):
    for item in os.listdir(path):
        nome_bruto = os.path.join(path, item)
        segmentado = item.split("-", 1)     
        novo_nome = segmentado[1]
        nome_limpo = os.path.join(path, novo_nome)   
        os.rename(nome_bruto, nome_limpo)

tirar_id(test_path)
tirar_id(valid_path)
tirar_id(train_path)

<h2>Transformações com Data Augmentation</h2>

In [5]:
train_transform = transforms.Compose([transforms.Resize((320, 320)),transforms.RandomCrop(299), transforms.RandomHorizontalFlip(),
                                      transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), ## varia brilho, contraste, saturação e matiz
                                      transforms.RandomRotation(10), transforms.ToTensor(),
                                      transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])])
## obs: o conjunto de teste não recebe o data augmentation, só a normalização.

test_transform = transforms.Compose([transforms.Resize((299, 299)), transforms.ToTensor(),
                                     transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])])

train_dataset = ImageFolder(root=train_path, transform=train_transform)
valid_dataset = ImageFolder(root=valid_path, transform=test_transform)
test_dataset  = ImageFolder(root=test_path,  transform=test_transform)

train_load = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_load = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_load  = DataLoader(test_dataset,  batch_size=1,  shuffle=False)

images, labels = next(iter(train_load))
print("Batch shape:", images.shape, labels.shape)

Batch shape: torch.Size([32, 3, 299, 299]) torch.Size([32])


<h2>Inception V3</h2>

In [6]:
model = inception_v3(weights=Inception_V3_Weights.DEFAULT, aux_logits=True)
classes = 120
#Camada de saída principal e auxiliar
model.fc = nn.Linear(model.fc.in_features, classes)
model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, classes)

In [15]:
for param in model.parameters():
    param.requires_grad = False

# liberar só as camadas de saída para treino
for param in model.fc.parameters():
    param.requires_grad = True
for param in model.AuxLogits.parameters():
    param.requires_grad = True

In [16]:
print(model)

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

<h2>Configurações do treino</h2>

In [9]:
epocas = 120
aprendizado = 1e-3
momentum = 0.9
paciencia= 10

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=aprendizado, momentum=momentum,weight_decay=1e-4) #Adição do weight_decay para L2 regularization

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epocas, eta_min=1e-6) ## taxa de aprendizado adaptativa

csv_file_path = "training_metrics.csv"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model  = model.to(device)

pd.DataFrame(columns=["Epoch", "Loss", "Accuracy", "LR"]).to_csv(csv_file_path, index=False)

cpu


In [22]:
last_val_acc= 0.0
for epoch in range(epocas):
    if paciencia == 0:
        break
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in train_load:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss1 = criterion(outputs.logits, targets)
        loss2 = criterion(outputs.aux_logits, targets)
        loss = loss1 + 0.3 * loss2
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.logits.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, targets in val_load:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            outputs = outputs.logits
            loss = criterion(outputs, targets)

            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += targets.size(0)
            val_correct += predicted.eq(targets).sum().item()

    val_loss /= len(val_dataset)
    val_accuracy = 100. * val_correct / val_total
    current_val_acc = val_accuracy
    if last_val_acc >= current_val_acc:   #Early stopping
        paciencia= paciencia-1
    last_val_acc = current_val_acc

    scheduler.step()  ## atualizar lr ao fim de cada época

    train_loss = running_loss / len(train_dataset)
    train_accuracy = 100. * correct / total
    current_lr = scheduler.get_last_lr()[0]

    pd.DataFrame([[epoch + 1, train_loss, train_accuracy, val_loss, val_accuracy, current_lr]],
                 columns=["Epoch", "Train_Loss", "Train_Accuracy","Val_Loss", "Val_Accuracy", "LR"]).to_csv(csv_file_path, mode='a', header=False, index=False)

    print(f"Epoch {epoch+1:3d}/{epocas} | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_accuracy:.2f}% |  Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2f}% | "f"LR: {current_lr:.2e}")
    
torch.save(model.state_dict(), "Modelo_Inception.pth")

'\nlast_val_acc= 0.0\nfor epoch in range(epocas):\n    if paciencia == 0:\n        break\n    model.train()\n    running_loss = 0.0\n    correct = 0\n    total = 0\n\n    for inputs, targets in train_load:\n        inputs, targets = inputs.to(device), targets.to(device)\n        optimizer.zero_grad()\n\n        outputs = model(inputs)\n        loss1 = criterion(outputs.logits, targets)\n        loss2 = criterion(outputs.aux_logits, targets)\n        loss = loss1 + 0.3 * loss2\n        loss.backward()\n        optimizer.step()\n\n        running_loss += loss.item() * inputs.size(0)\n        _, predicted = outputs.logits.max(1)\n        total += targets.size(0)\n        correct += predicted.eq(targets).sum().item()\n\n    model.eval()\n    val_loss = 0.0\n    val_correct = 0\n    val_total = 0\n\n    with torch.no_grad():\n        for inputs, targets in val_load:\n            inputs, targets = inputs.to(device), targets.to(device)\n\n            outputs = model(inputs)\n            outpu

<h2>Métricas do treino</h2>

<h2>Avaliação do Modelo</h2>

<h2>Conclusões</h2>